# API de Anthropic - Parámetros Completos

Demostración interactiva de todos los parámetros disponibles en la API Messages de Anthropic.

**Referencia:** https://platform.claude.com/docs/en/api/messages

In [ ]:
import os
from typing import Optional, Dict, Any
from dotenv import load_dotenv
from anthropic import Anthropic
from google.colab import userdata
import json

# Load environment variables
load_dotenv()

# Retrieve API key from Colab secrets
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
client = Anthropic(api_key=ANTHROPIC_API_KEY)
print("✅ Cliente Anthropic inicializado correctamente")

## 1. Parámetros Básicos

### temperature: Control de aleatoriedad

In [ ]:
# Deterministic (temperature = 0.0)
print("\n" + "="*70)
print("Temperature = 0.0 (Determinístico)")
print("="*70)

for i in range(2):
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=50,
        temperature=0.0,
        messages=[{"role": "user", "content": "Completa: El cielo es..."}]
    )
    print(f"Intento {i+1}: {response.content[0].text}")

In [ ]:
# Creative (temperature = 0.9)
print("\n" + "="*70)
print("Temperature = 0.9 (Creativo)")
print("="*70)

for i in range(2):
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=80,
        temperature=0.9,
        messages=[{"role": "user", "content": "Escribe una línea poética sobre la programación"}]
    )
    print(f"Intento {i+1}:\n{response.content[0].text}\n")

### top_p: Nucleus Sampling

Solo considera tokens cuya probabilidad acumulada es ≤ top_p

In [ ]:
print("\n" + "="*70)
print("top_p = 0.9 (Conservador)")
print("="*70)

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=100,
    top_p=0.9,
    messages=[{"role": "user", "content": "¿Cuál es la capital de España?"}]
)

print(f"Respuesta: {response.content[0].text}")
print(f"Tokens entrada: {response.usage.input_tokens}")
print(f"Tokens salida: {response.usage.output_tokens}")

### top_k: Limitar tokens candidatos

Solo elige de los k tokens más probables

In [ ]:
# top_k = 1 (solo el token más probable)
print("\n" + "="*70)
print("top_k = 1 (Extremadamente determinístico)")
print("="*70)

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=50,
    top_k=1,
    messages=[{"role": "user", "content": "¿Cuánto es 2+2?"}]
)

print(f"Respuesta: {response.content[0].text}")

## 2. Parámetros de Control: stop_sequences

Detener generación cuando se encuentre una secuencia específica

In [ ]:
print("\n" + "="*70)
print("Sin stop_sequences")
print("="*70)

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=200,
    system="Eres un asistente. Numbera cada punto.",
    messages=[{"role": "user", "content": "Listame 5 beneficios de Python"}]
)

print(response.content[0].text)

In [ ]:
print("\n" + "="*70)
print("Con stop_sequences = ['\\n4.']")
print("="*70)

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=200,
    system="Eres un asistente. Numbera cada punto.",
    stop_sequences=["\n4."],
    messages=[{"role": "user", "content": "Listame 5 beneficios de Python"}]
)

print(response.content[0].text)
print(f"\n⏹️  Razón de parada: {response.stop_reason}")

## 3. System Prompt: Instrucciones del Sistema

Guiar el comportamiento del modelo sin contar tokens del usuario

In [ ]:
# Sin system prompt
print("\n" + "="*70)
print("SIN System Prompt")
print("="*70)

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=150,
    messages=[{"role": "user", "content": "¿Cuáles son las 3 ciudades más grandes de España?"}]
)

print("Respuesta:")
print(response.content[0].text)

In [ ]:
# Con system prompt
print("\n" + "="*70)
print("CON System Prompt (respuestas ultra concisas)")
print("="*70)

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=150,
    system="Responde de forma ultra concisa. Máximo 2 frases. Solo hechos, sin explicaciones.",
    messages=[{"role": "user", "content": "¿Cuáles son las 3 ciudades más grandes de España?"}]
)

print("Respuesta:")
print(response.content[0].text)

## 4. Metadata: Tracking y Debugging

Información personalizada que se retorna en la respuesta

In [ ]:
print("\n" + "="*70)
print("Con Metadata para tracking")
print("="*70)

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=100,
    metadata={
        "session_id": "session_001",
        "user_id": "user_123",
        "feature_test": "parameters_demo",
        "version": "1.0"
    },
    messages=[{"role": "user", "content": "¿Quién eres?"}]
)

print(f"Respuesta: {response.content[0].text}")
print(f"\n✅ Metadata enviado fue retornado en la respuesta")

## 5. Comparación de Parámetros de Muestreo

Distintas combinaciones para diferentes casos de uso

In [ ]:
prompt = "Describe un día en la vida de un programador de IA"

configs = [
    {"name": "Determinístico", "temperature": 0.0, "top_k": 1},
    {"name": "Conservador", "temperature": 0.3, "top_p": 0.9},
    {"name": "Balance", "temperature": 0.7, "top_p": 0.95},
    {"name": "Creativo", "temperature": 0.9, "top_p": 0.99},
]

for config in configs:
    print("\n" + "="*70)
    print(f"Configuración: {config['name']}")
    print(f"Parameters: {json.dumps({k:v for k,v in config.items() if k != 'name'}, indent=2)}")
    print("="*70)
    
    kwargs = {k: v for k, v in config.items() if k != 'name'}
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=120,
        **kwargs,
        messages=[{"role": "user", "content": prompt}]
    )
    
    print(response.content[0].text[:200] + "..." if len(response.content[0].text) > 200 else response.content[0].text)

## 6. Tokens y Costos

Monitorear uso de tokens y calcular costos

In [ ]:
# Pricing per model (input/output per 1M tokens)
PRICING = {
    "claude-haiku-4-5-20251001": {"input": 0.80, "output": 4.00},
    "claude-sonnet-4-6": {"input": 3.00, "output": 15.00},
    "claude-opus-4-7": {"input": 15.00, "output": 75.00},
}

responses_data = []

models = ["claude-haiku-4-5-20251001", "claude-sonnet-4-6", "claude-opus-4-7"]
test_prompt = "Explica qué es la programación en Python"

for model in models:
    response = client.messages.create(
        model=model,
        max_tokens=150,
        messages=[{"role": "user", "content": test_prompt}]
    )
    
    pricing = PRICING[model]
    input_cost = (response.usage.input_tokens / 1_000_000) * pricing["input"]
    output_cost = (response.usage.output_tokens / 1_000_000) * pricing["output"]
    total_cost = input_cost + output_cost
    
    responses_data.append({
        "model": model,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
        "total_tokens": response.usage.input_tokens + response.usage.output_tokens,
        "cost_usd": total_cost
    })

# Display comparison
import pandas as pd
df = pd.DataFrame(responses_data)
print("\n" + "="*70)
print("Comparación de Modelos - Tokens y Costos")
print("="*70)
print(df.to_string(index=False))
print(f"\n💰 Modelo más económico: {df.loc[df['cost_usd'].idxmin(), 'model']} (${df['cost_usd'].min():.6f})")

## 7. Multi-turn Conversation

Manteniendo contexto entre múltiples mensajes

In [ ]:
print("\n" + "="*70)
print("Conversación Multi-turno")
print("="*70)

messages = [
    {"role": "user", "content": "Me llamo Juan y soy ingeniero de software"},
    {"role": "assistant", "content": "Mucho gusto Juan. Soy Claude, un asistente de IA. Es interesante que seas ingeniero de software. ¿En qué área específica trabajas?"},
    {"role": "user", "content": "Me especializo en machine learning y procesamiento de datos"},
]

# Nuevo turno
print("\nUsuario: ¿Qué me recomiendas aprender?")

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=200,
    system="Eres un mentor de tecnología. Sé conciso y práctico.",
    messages=messages + [{"role": "user", "content": "¿Qué me recomiendas aprender?"}]
)

print(f"\nClaude: {response.content[0].text}")
print(f"\nTokens entrada: {response.usage.input_tokens}")
print(f"Tokens salida: {response.usage.output_tokens}")

## Conclusiones

### Parámetros Clave según Caso de Uso:

| Caso de Uso | Temperature | top_p | top_k | Recomendación |
|------------|------------|--------|--------|---------------|
| Análisis/Coding | 0.0-0.3 | 0.9 | - | Determinístico y preciso |
| Resúmenes | 0.3-0.5 | 0.95 | - | Balance entre calidad y consistencia |
| Escritura creativa | 0.7-0.9 | 0.95-0.99 | - | Más variabilidad |
| Brainstorming | 0.8-1.0 | 0.98-1.0 | - | Máxima creatividad |
| Respuestas rápidas | 0.3-0.5 | 0.9 | 1 | Eficiente y consistente |

### Tips de Optimización:

1. **Costo**: Usa Haiku para tareas simples, Opus solo cuando sea necesario
2. **Velocidad**: Temperature baja → más rápido
3. **Calidad**: Buenos system prompts > parámetros mágicos
4. **Control**: Usa stop_sequences para limitar formato
5. **Tracking**: Usa metadata para debugging y análisis